# TF-IDF + Logistic Regression vs Fine-Tuned BERT

This notebook compares two **trained models for the same task**:

1. **Traditional baseline:** TF-IDF text features with multinomial logistic regression.
2. **Improved model:** the saved fine-tuned BERT classifier in `bert_symptom_model`.

Both models are evaluated on the same 240 held-out symptom descriptions from `Symptom2Disease.csv`. The exact `random_state=42` split from `Fine Tune Bert.ipynb` is recreated so the saved BERT model remains comparable.

## 1. Install dependencies

Run the next line only if the required packages are missing.

In [19]:
# Uncomment if needed:
# %pip install pandas numpy scikit-learn torch transformers

## 2. Imports and reproducibility

In [20]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 3. Load and split the dataset

The saved BERT model was trained using an 80/20 split with `random_state=42` and without stratification. This notebook recreates that exact split. Changing the split now would evaluate BERT on rows that may have been used during its training.

In [21]:
DATA_PATH = Path("Symptom2Disease.csv")
FINE_TUNED_MODEL_PATH = Path("bert_symptom_model")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")
if not FINE_TUNED_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Fine-tuned model folder not found: {FINE_TUNED_MODEL_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH).dropna(subset=["label", "text"]).copy()
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
class_names = label_encoder.classes_.tolist()
num_labels = len(class_names)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Full dataset: {len(df)} rows")
print(f"Training set: {len(train_df)} rows")
print(f"Held-out test set: {len(test_df)} rows")
print(f"Number of disease classes: {num_labels}")

Full dataset: 1200 rows
Training set: 960 rows
Held-out test set: 240 rows
Number of disease classes: 24


## 4. Train the TF-IDF + logistic regression baseline

The scikit-learn pipeline prevents data leakage: TF-IDF vocabulary and weights are learned only from the training descriptions. Word unigrams and bigrams allow the baseline to represent individual terms and short symptom phrases.

In [22]:
tfidf_logistic_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                max_features=20_000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2_000,
                random_state=SEED,
            ),
        ),
    ]
)

tfidf_logistic_model.fit(
    train_df["text"].astype(str),
    train_df["label_id"],
)

tfidf_predictions = tfidf_logistic_model.predict(
    test_df["text"].astype(str)
)

feature_count = len(
    tfidf_logistic_model.named_steps["tfidf"].get_feature_names_out()
)
print(f"TF-IDF features learned from training data: {feature_count}")
print("TF-IDF + logistic regression training completed.")

TF-IDF features learned from training data: 7738
TF-IDF + logistic regression training completed.


## 5. Load and evaluate the saved fine-tuned BERT

No additional BERT training occurs in this notebook. The saved model is evaluated on the same test descriptions used for the traditional baseline.

In [23]:
bert_tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED_MODEL_PATH)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    FINE_TUNED_MODEL_PATH
).to(device)
bert_model.eval()

if bert_model.config.num_labels != num_labels:
    raise ValueError(
        "The saved BERT output size does not match the dataset: "
        f"model={bert_model.config.num_labels}, dataset={num_labels}"
    )


def predict_bert_in_batches(texts, batch_size=16, max_length=128):
    predictions = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        encoded = bert_tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {name: value.to(device) for name, value in encoded.items()}

        with torch.inference_mode():
            logits = bert_model(**encoded).logits

        predictions.extend(logits.argmax(dim=1).cpu().numpy().tolist())

    return np.asarray(predictions)


test_texts = test_df["text"].astype(str).tolist()
y_true = test_df["label_id"].to_numpy()
bert_predictions = predict_bert_in_batches(test_texts)

print("Fine-tuned BERT predictions completed.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4126.45it/s]


Fine-tuned BERT predictions completed.


## 6. Compare both models

- **Accuracy:** overall proportion of correct predictions.
- **Macro metrics:** give every disease class equal importance.
- **Weighted metrics:** weight each disease class by its number of test examples.

In [24]:
def calculate_metrics(model_name, labels, predictions):
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(labels, predictions),
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Weighted Precision": weighted_precision,
        "Weighted Recall": weighted_recall,
        "Weighted F1": weighted_f1,
    }


comparison = pd.DataFrame(
    [
        calculate_metrics(
            "TF-IDF + Logistic Regression",
            y_true,
            tfidf_predictions,
        ),
        calculate_metrics(
            "Fine-tuned BERT",
            y_true,
            bert_predictions,
        ),
    ]
)

metric_columns = comparison.columns.drop("Model")
comparison_percent = comparison.copy()
comparison_percent[metric_columns] = (
    comparison_percent[metric_columns].mul(100).round(2)
)

print("All metric values below are percentages (%).")
display(comparison_percent)

All metric values below are percentages (%).


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0,TF-IDF + Logistic Regression,96.67,97.15,96.78,96.70,96.99,96.67,96.54
1,Fine-tuned BERT,87.92,88.27,87.62,86.94,88.58,87.92,87.25


## 7. Improvement of TF-IDF + logistic regression over fine-tuned BERT

Positive values show how many percentage points the TF-IDF baseline scored above fine-tuned BERT. The underlying model scores are unchanged; only the direction of subtraction is selected to match the primary model.

In [25]:
tfidf_improvement = (
    comparison_percent.loc[0, metric_columns]
    - comparison_percent.loc[1, metric_columns]
)

improvement_table = pd.DataFrame(
    {
        "Metric": metric_columns,
        "TF-IDF improvement (percentage points)": tfidf_improvement.values,
    }
)

display(improvement_table)

,Metric,TF-IDF improvement (percentage points)
0,Accuracy,8.75
1,Macro Precision,8.88
2,Macro Recall,9.16
3,Macro F1,9.76
4,Weighted Precision,8.41
5,Weighted Recall,8.75
6,Weighted F1,9.29


## 8. Per-class reports

In [26]:
all_label_ids = np.arange(num_labels)

tfidf_report = pd.DataFrame(
    classification_report(
        y_true,
        tfidf_predictions,
        labels=all_label_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

bert_report = pd.DataFrame(
    classification_report(
        y_true,
        bert_predictions,
        labels=all_label_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

print("TF-IDF + logistic regression class-level report")
display(tfidf_report.round(4))

print("Fine-tuned BERT class-level report")
display(bert_report.round(4))

TF-IDF + logistic regression class-level report


,precision,recall,f1-score,support
Acne,1.0000,1.0000,1.0000,7.0000
Arthritis,1.0000,1.0000,1.0000,10.0000
Bronchial Asthma,1.0000,1.0000,1.0000,11.0000
Cervical spondylosis,1.0000,1.0000,1.0000,7.0000
Chicken pox,0.8571,1.0000,0.9231,12.0000
Common Cold,0.9231,1.0000,0.9600,12.0000
Dengue,1.0000,0.6667,0.8000,12.0000
Dimorphic Hemorrhoids,1.0000,1.0000,1.0000,7.0000
Fungal infection,1.0000,1.0000,1.0000,13.0000
Hypertension,1.0000,1.0000,1.0000,10.0000


Fine-tuned BERT class-level report


,precision,recall,f1-score,support
Acne,1.0000,1.0000,1.0000,7.0000
Arthritis,1.0000,1.0000,1.0000,10.0000
Bronchial Asthma,1.0000,1.0000,1.0000,11.0000
Cervical spondylosis,0.8750,1.0000,0.9333,7.0000
Chicken pox,1.0000,0.5833,0.7368,12.0000
Common Cold,0.9231,1.0000,0.9600,12.0000
Dengue,0.7273,0.6667,0.6957,12.0000
Dimorphic Hemorrhoids,1.0000,1.0000,1.0000,7.0000
Fungal infection,1.0000,0.9231,0.9600,13.0000
Hypertension,0.9091,1.0000,0.9524,10.0000


## 9. Save thesis-ready results

In [27]:
comparison_percent.to_csv("tfidf_vs_finetuned_bert.csv", index=False)
improvement_table.to_csv("tfidf_improvement_over_bert.csv", index=False)
tfidf_report.to_csv("tfidf_logistic_regression_classification_report.csv")
bert_report.to_csv("fine_tuned_bert_classification_report.csv")

print("Saved:")
print("- tfidf_vs_finetuned_bert.csv")
print("- tfidf_improvement_over_bert.csv")
print("- tfidf_logistic_regression_classification_report.csv")
print("- fine_tuned_bert_classification_report.csv")

Saved:
- tfidf_vs_finetuned_bert.csv
- tfidf_improvement_over_bert.csv
- tfidf_logistic_regression_classification_report.csv
- fine_tuned_bert_classification_report.csv


## Suggested thesis interpretation

TF-IDF with logistic regression is a meaningful trained baseline because it learns from the same labeled training set and predicts the same 24 disease classes. Fine-tuned BERT should be described as better only when its measured test metrics exceed the baseline. If the traditional baseline performs similarly or better, report that result honestly and discuss the small, balanced, and highly structured nature of the dataset.